In [1]:
from manim import *
import numpy as np
import csv
import os

Manim Community v0.18.0

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
#from manim_voicevox_util import MVUtil

In [10]:
Tex.set_default(tex_template=TexTemplate(
    tex_compiler = "lualatex", 
    # tex_compiler = "luatex" でも可
    output_format = ".pdf", 
    preamble = r"""
        \usepackage{amsmath}
        \usepackage{amssymb}
        \usepackage{luatexja}
        \usepackage[haranoaji]{luatexja-preset}
    """
))

In [ ]:
%%manim -qm -v WARNING -o no_voice_video.mp4 Test 

class Test(Scene):
    def construct(self):
        #if os.path.isfile(MVUtil.timestamp_path):
        #    os.remove(MVUtil.timestamp_path)

        axes = Axes(
            x_range=[-1.1, 1.1, 1],
            y_range=[-1.1, 1.1, 1],
            x_length=6,
            y_length=6,
            tips=False,
        ).add_coordinates()

        labels = axes.get_axis_labels(
            x_label=Tex("$x$"), y_label=Tex("$y$")
        )

        self.add(axes, labels)

        n_list = [1, 2, 3, 10, 100]

        for n in n_list:
            n_label = Tex(f'$n={n}$').scale(1.5).to_edge(UL)
            #MVUtil.add_subtitle(self, 74, f'$n={n}$の場合', Write(n_label))

            self.animate_polygon_and_arrow(n, axes)

            self.play(Unwrite(n_label))
            self.wait(0.5)
    
    def animate_polygon_and_arrow(self, n: int, axes: Axes):
        m = 2 * n + 1
        theta = np.pi / (2.0 * n + 1.0)

        graph = self.make_graph(n)
        polygon = Polygon(*axes.c2p(graph), color=WHITE)
        
        self.play(
            Create(polygon, run_time=3, rate_func=rate_functions.ease_in_out_sine)
        )
        self.wait(0.5)

        line_start = np.array([0, 1, 0])
        line_end = np.array([0, 0, 0])
        line = Arrow(*axes.c2p([line_start, line_end]), color=RED, buff=0)

        self.play(Create(line, run_time=1))

        line_animate_group = []
        
        line_animate_group.append(ApplyMethod(line.shift, *axes.c2p(graph[0, :] - line_start)))

        line_animate_group.append(Rotate(line, -theta / 2.0, about_point=axes.c2p(graph[0, :])[0]))

        now_vertex_i = 0
        
        for i in range(m):
            next_vertex_i = (now_vertex_i + n) % m

            move_vector = graph[2 * next_vertex_i, :] - graph[2 * now_vertex_i, :]
            move_vector_len = np.linalg.norm(move_vector)
            move_vector *= (move_vector_len - 1) / move_vector_len

            line_animate_group.append(ApplyMethod(line.shift, *axes.c2p(move_vector)))
            line_animate_group.append(Wait(run_time=0.1))

            now_vertex_i = next_vertex_i
        
            if now_vertex_i != 0:
                line_animate_group.append(Rotate(line, -theta, about_point=axes.c2p(graph[2 * now_vertex_i, :])[0]))
            else:
                line_animate_group.append(Rotate(line, -theta / 2.0, about_point=axes.c2p(graph[2 * now_vertex_i, :])[0]))
        
        self.play(Succession(*line_animate_group), run_time=8, rate_func=rate_functions.ease_in_out_sine)

        self.wait(0.5)
        self.play(AnimationGroup(*[Uncreate(polygon), Uncreate(line)]), run_time=1)


    def make_graph(self, n: int) -> np.ndarray:
        m = 2 * n + 1

        theta = np.pi / (2.0 * n + 1.0)
        outer_edge = np.sin(theta / 2.0) / np.sin(theta)

        rotation_matrix = np.array([
            [np.cos(2 * theta), -np.sin(2 * theta), 0],
            [np.sin(2 * theta), np.cos(2 * theta), 0],
            [0, 0, 1]
        ])

        half_rotation_matrix = np.array([
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta), np.cos(theta), 0],
            [0, 0, 1]
        ])

        outer_polygon_vertex = np.array([0, outer_edge * (np.sin(theta / 2.0) * np.tan(((2 * n - 1) / (2 * n + 1)) * np.pi / 2.0) + np.cos(theta / 2.0)), 0])

        outer_polygon_graph = self.make_polygon_graph(outer_polygon_vertex, rotation_matrix, m)

        inner_polygon_vertex = np.array([0, outer_edge * np.sin(theta / 2.0) / np.cos(((2 * n - 1) / (2 * n + 1)) * np.pi / 2.0), 0])
        inner_polygon_vertex = half_rotation_matrix @ inner_polygon_vertex

        inner_polygon_graph = self.make_polygon_graph(inner_polygon_vertex, rotation_matrix, m)

        graph = []

        for i in range(m):
            graph.extend([outer_polygon_graph[i, :], inner_polygon_graph[i, :]])

        graph = np.array(graph)

        return graph

    def make_polygon_graph(self, vertex: np.ndarray, rotation_matrix: np.ndarray, vertex_n: int):
        graph = [vertex]

        for i in range(vertex_n):
            vertex = rotation_matrix @ vertex
            graph.append(vertex)

        graph = np.array(graph)

        return graph


In [12]:
# from pydub import AudioSegment

# movie_time = 0
# all_sound = AudioSegment.silent(duration=0)

# with open(MVUtil.timestamp_path) as f:
#     reader = csv.reader(f)

#     for file_time, file_path in reader:
#         file_time = float(file_time)
#         silent_time = file_time - movie_time
        
#         silent = AudioSegment.silent(duration=silent_time * 1000)
#         voice = AudioSegment.from_file(file_path)

#         all_sound += silent + voice

#         movie_time = file_time + voice.duration_seconds

# all_sound.export('all_sound.wav', format='wav')

In [13]:
import moviepy.editor as mp

video = mp.VideoFileClip('./media/videos/manim/720p30/no_voice_video.mp4').subclip()
video = video.set_audio(mp.AudioFileClip('all_sound.wav'))
video.write_videofile('correct_video.mp4', audio_codec='aac')

Moviepy - Building video correct_video.mp4.
MoviePy - Writing audio in correct_videoTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video correct_video.mp4



Moviepy - Done !
Moviepy - video ready correct_video.mp4


Scene内で関数constructを用意してそれが実行され、動画ファイルとなる
voicevoxと連携するためには、音声書き出しもconstruct内で行って、音声の長さなどを確認する必要がある
conda install https://github.com/VOICEVOX/voicevox_core/releases/download/0.15.0/voicevox_core-0.15.0+cuda-cp38-abi3-win_amd64.whl
pip install voicevox_core-0.15.0+cuda-cp38-abi3-win_amd64.whl

In [14]:
# from manim_voicevox_util import LatexUtil
# from manim_voicevox_util import VoicevoxWrapper

In [15]:
# VoicevoxWrapper.create_wav_from_latex(2, 'もう1つの式、$z=2$')


In [16]:
# from pydub import AudioSegment

In [17]:
# silent = AudioSegment.silent(duration=3000)
# silent.export("test.wav",format="wav")